In [1]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
import random
from time import sleep
import pandas as pd
from dotenv import load_dotenv
import os
import requests
import unicodedata

In [2]:
load_dotenv()
WEB_BASE = os.getenv("WEB_BASE")
WEB_MANUFACTURER = os.getenv("WEB_MANUFACTURER")
WEB_MODELS = os.getenv("WEB_MODELS")
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json"
}

In [ ]:
url_marcas = WEB_MANUFACTURER
# print(url_marcas)

https://www.compramostucoche.es/papi/v1/car-types/manufacturer


In [5]:
marcas_ordenadas = sorted([m['nombre'] for m in catalogo], key=len, reverse=True)

mapa_modelos = {m['nombre']: sorted(m['modelos'], key=len, reverse=True) for m in catalogo}

def normalizar(texto):
    if not texto: return ""
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto)
                  if unicodedata.category(c) != 'Mn')
    return texto.lower().strip()

marcas_normalizadas = {normalizar(m): m for m in marcas_ordenadas}

mapa_modelos_norm = {}
for marca_real, modelos in mapa_modelos.items():
    mapa_modelos_norm[normalizar(marca_real)] = {normalizar(mod): mod for mod in modelos}

In [6]:
mapa_modelos_norm

{'abarth': {'grande punto': 'Grande Punto',
  '500e cabrio': '500e Cabrio',
  '124 spider': '124 Spider',
  'punto': 'Punto',
  'ritmo': 'Ritmo',
  'stilo': 'Stilo',
  '500e': '500e',
  '595c': '595C',
  '600e': '600e',
  '695c': '695C',
  '500': '500',
  '595': '595',
  '695': '695'},
 'aiways': {'u5': 'U5', 'u6': 'U6'},
 'aixam': {'crossline emotion': 'Crossline Emotion',
  'crossover emotion': 'Crossover Emotion',
  'coupe emotion': 'Coupe Emotion',
  'city emotion': 'City Emotion',
  'crossover': 'Crossover',
  'sensation': 'Sensation'},
 'alfa romeo': {'alfetta serie iii/gtv': 'Alfetta Serie III/GTV',
  '8c competizione': '8C Competizione',
  'alfa spider': 'Alfa Spider',
  'alfa brera': 'Alfa Brera',
  '4c spider': '4C Spider',
  'giulietta': 'Giulietta',
  'alfa 145': 'Alfa 145',
  'alfa 146': 'Alfa 146',
  'alfa 147': 'Alfa 147',
  'alfa 155': 'Alfa 155',
  'alfa 156': 'Alfa 156',
  'alfa 159': 'Alfa 159',
  'alfa 164': 'Alfa 164',
  'alfa 166': 'Alfa 166',
  'alfa gtv': 'Alfa 

In [10]:
driver = webdriver.Chrome()

In [11]:
lista_coches = []

In [12]:
for page in range(1, 341):
    url = f'{WEB_BASE}comprar-coche/?page={page}'
    
    try:
        driver.get(url)
        sleep(random.uniform(2, 4)) 

        cars = driver.find_elements(By.CLASS_NAME, 'root___Dz4kU')

        for car in cars:
            try:
        # 1. RESETEO TOTAL
                marca_final, modelo_final, version_final = "Desconocida", "Desconocido", ""
                
                nombre_completo = car.find_element(By.CLASS_NAME, 'title___uRijL').text
                nombre_norm = normalizar(nombre_completo)

                # 2. BUSCAR MARCA
                for m_norm, m_real in marcas_normalizadas.items():
                    if m_norm in nombre_norm: # 'in' es mejor que 'startswith'
                        marca_final = m_real
                        # El resto del texto donde buscaremos el modelo
                        texto_busqueda_modelo = nombre_norm.replace(m_norm, "").strip()
                        
                        modelos_dict = mapa_modelos_norm.get(m_norm, {})
                        # Ordenar por longitud (descendente) para no confundir "Clase E" con "Clase E Coupe"
                        modelos_ordenados = sorted(modelos_dict.items(), key=lambda x: len(x[0]), reverse=True)

                        # 3. BUSCAR MODELO (Estrategia Reforzada)
                        texto_busqueda_modelo = nombre_norm.replace(m_norm, "").strip()
                        # Creamos una versión "ultra-limpia" del texto del anuncio (sin guiones ni símbolos)
                        texto_ultra_limpio = texto_busqueda_modelo.replace("-", " ").replace("_", " ")

                        for mod_norm, mod_real in modelos_ordenados:
                            # 1. Probamos coincidencia directa
                            # 2. Probamos reemplazando guiones por espacios (ej: C-Class -> C Class)
                            mod_norm_espacios = mod_norm.replace("-", " ")
                            
                            # Caso especial para Mercedes/BMW: Si el modelo es "clase a", probamos a buscar solo " a "
                            mod_abreviado = mod_norm.replace("clase", "").replace("serie", "").strip()

                            if (mod_norm in texto_busqueda_modelo or 
                                mod_norm_espacios in texto_ultra_limpio or
                                (len(mod_abreviado) > 0 and f" {mod_abreviado} " in f" {texto_ultra_limpio} ")):
                                
                                modelo_final = mod_real
                                version_final = nombre_completo.replace(m_real, "").replace(mod_real, "").strip()
                                break

                        # --- Si después de todo NO SE ENCUENTRA, probamos búsqueda inversa ---
                        if modelo_final == "Desconocido":
                            for mod_norm, mod_real in modelos_ordenados:
                                # A veces el modelo del catálogo es más largo que el del anuncio
                                # Si "Golf" está en "Golf VII", lo aceptamos
                                if len(mod_norm) > 2 and mod_norm in texto_busqueda_modelo:
                                    modelo_final = mod_real
                                    break

                # --- LOG DE DIAGNÓSTICO ---
                if modelo_final == "Desconocido":
                    print(f"⚠️ OMITIDO (Modelo no encontrado): {nombre_completo}")
                    continue

                # 4. EXTRACCIÓN DE ATRIBUTOS (con selectores específicos)
                km = car.find_element(By.CSS_SELECTOR, '[data-qa-selector="mileage"]').text
                power_text = car.find_element(By.CSS_SELECTOR, '[data-qa-selector="horsePower"]').text
                price = car.find_element(By.CSS_SELECTOR, '[data-qa-selector="price"]').text

                # Limpieza segura de potencia
                # Entrada: "110 kW (150 CV)" -> Salida: 150
                try:
                    power_cv = power_text.split('(')[1].split(' ')[0]
                except:
                    power_cv = 0

                datos_coche = {
                    "marca": marca_final,
                    "modelo": modelo_final,
                    "version": version_final,
                    "registration": car.find_element(By.CSS_SELECTOR, '[data-qa-selector="registration"]').text,
                    "km": int(km.replace('.', '').replace(' km', '')),
                    "gear_type": car.find_element(By.CSS_SELECTOR, '[data-qa-selector="transmission"]').text,
                    "fuel_type": car.find_element(By.CSS_SELECTOR, '[data-qa-selector="fuelType"]').text,
                    "power": int(power_cv),
                    "price": int(price.replace('.', '').replace(' €', ''))
                }
                
                lista_coches.append(datos_coche)

            except Exception as e:
                print(f"Coche omitido en {nombre_completo}. Error: {type(e).__name__}")
                continue
                
    except Exception as e:
        print(f"Error en la página {page}: {e}")
        break

⚠️ OMITIDO (Modelo no encontrado): Mercedes-Benz Clase A A 250e
⚠️ OMITIDO (Modelo no encontrado): Mercedes-Benz Clase B B 200 CDI
⚠️ OMITIDO (Modelo no encontrado): Mercedes-Benz Clase GLC GLC 200 d
⚠️ OMITIDO (Modelo no encontrado): MINI MINI Cooper
⚠️ OMITIDO (Modelo no encontrado): Mercedes-Benz Clase B B 200
⚠️ OMITIDO (Modelo no encontrado): BMW Serie 3 318i
⚠️ OMITIDO (Modelo no encontrado): MINI MINI Cooper
⚠️ OMITIDO (Modelo no encontrado): Citroen C 3 1.5 Blue-HDi
⚠️ OMITIDO (Modelo no encontrado): Mercedes-Benz Clase B B 180 CDI
⚠️ OMITIDO (Modelo no encontrado): Mercedes-Benz Clase A A 200 d
⚠️ OMITIDO (Modelo no encontrado): BMW Serie 5 530d
⚠️ OMITIDO (Modelo no encontrado): Citroen C 3 1.2 PureTech
⚠️ OMITIDO (Modelo no encontrado): BMW Serie 2 225xe Active Tourer
⚠️ OMITIDO (Modelo no encontrado): Citroen C 3 1.2 PureTech
⚠️ OMITIDO (Modelo no encontrado): Citroen C 3 1.2 PureTech
⚠️ OMITIDO (Modelo no encontrado): Mercedes-Benz Clase A A 180
⚠️ OMITIDO (Modelo no encon

KeyboardInterrupt: 

In [ ]:
df = pd.DataFrame(lista_coches)
len(df)

3035

In [ ]:
df

,marca,modelo,version,registration,km,gear_type,fuel_type,power,price
0,Peugeot,3008,1.2 PureTech,07/2017,133864,Manual,Gasolina,130,10499
1,Peugeot,3008,1.5 Blue-HDi,04/2019,86230,Manual,Diésel,130,13699
2,Ford,Fiesta,1.0 EcoBoost,02/2020,65559,Manual,Gasolina,95,13299
3,Peugeot,2008,1.5 Blue-HDi,06/2020,94913,Automático,Diésel,131,15399
4,Audi,Q7,3.0 V6 TDI,11/2015,147531,Automático,Diésel,272,29699
...,...,...,...,...,...,...,...,...,...
3030,Opel,Insignia Grand Sport,1.5 SIDI Turbo,05/2018,33574,Automático,Gasolina,165,17199
3031,Hyundai,i10,1.0,06/2022,37545,Manual,Gasolina,67,12399
3032,Nissan,Qashqai,1.3 DIG-T Mild-Hybrid,01/2024,22849,Manual,Gasolina,140,23899
3033,Peugeot,108,1.0 VTi,04/2019,109253,Manual,Gasolina,72,8299


In [ ]:
df.to_csv('../datasets/data_compramostucoche.csv', index=False)

In [ ]:
marcas_bbdd = []
modelos_bbdd = []
contador_modelos = 1

for item in catalogo:
    m_id = item['id_marca']
    m_nombre = item['nombre']
    
    # 1. Añadimos a la lista de marcas
    marcas_bbdd.append({
        "id": m_id,
        "nombre": m_nombre
    })
    
    for id_mod, nombre_mod in item['modelos'].items():
        modelos_bbdd.append({
            "id": contador_modelos,
            "nombre": nombre_mod,
            "id_marca": m_id
        })
        contador_modelos += 1

In [ ]:
pd.DataFrame(marcas_bbdd).to_csv('../datasets/seed_marcas.csv', index=False)
pd.DataFrame(modelos_bbdd).to_csv('../datasets/seed_modelos.csv', index=False)